In [ ]:
import kagglehub
import os
import shutil
import yaml
from ultralytics import YOLO

# 1. download and copy dataset
print("downloading dataset...")
raw_path = kagglehub.dataset_download("yusufmurtaza01/grape-leaf-diseases")

colab_path = "/content/grape_dataset"
if os.path.exists(colab_path):
    shutil.rmtree(colab_path)
shutil.copytree(raw_path, colab_path)

# 2. Finding paths to val/train
train_dir = None
val_dir = None

for root, dirs, files in os.walk(colab_path):
    if 'train' in dirs:
        t_path = os.path.join(root, 'train')
        train_dir = os.path.join(t_path, 'images') if os.path.exists(os.path.join(t_path, 'images')) else t_path

    if 'valid' in dirs or 'val' in dirs:
        v_name = 'valid' if 'valid' in dirs else 'val'
        v_path = os.path.join(root, v_name)
        val_dir = os.path.join(v_path, 'images') if os.path.exists(os.path.join(v_path, 'images')) else v_path

print(f"Found path to train: {train_dir}")
print(f"Found path to val: {val_dir}")

# 3. Creating data-conf
yaml_file = os.path.join(colab_path, "data_fixed.yaml")
clean_config = {
    'train': train_dir,
    'val': val_dir,
    'nc': 4,
    'names': ['Grape__BlackRot', 'Grape__Esca', 'Grape__Healthy', 'Grape__LeafBlight']
}


with open(yaml_file, 'w') as f:
    yaml.dump(clean_config, f)

print("File data_fixed.yaml was created!")

In [ ]:
# train
model = YOLO("yolov8n.pt")
results = model.train(
    data=yaml_file,
    epochs=100,
    imgsz=640,
    batch=16,
    patience=15,
    project="grape_final",
    name="yolov8n_run2"
)

# metrics
map50 = results.box.map50
precision = results.box.mp
recall = results.box.mr
print(f"\nmAP50: {round(map50, 4)}")
print(f"Precision: {round(precision, 4)}")
print(f"Recall: {round(recall, 4)}")

In [ ]:
from ultralytics import YOLO

# save best weights
model = YOLO("/content/runs/detect/grape_final/yolov8n_run2/weights/best.pt")

# save as ONNX
model.export(format="onnx", simplify=True)

print("Model is saved as best.onnx!")

In [ ]:
#archive download
import shutil
shutil.make_archive('grape_dataset', 'zip', '/content/runs')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>